## Producer code

In [0]:
myconfigurations  = {
    "kafka.bootstrap.servers" : "pkc-12576z.us-west2.gcp.confluent.cloud:9092",
    "subscribe" : "first_topic" , 
    "startingOffsets" : "latest" , 
    "kafka.security.protocol" : "SASL_SSL" , 
    "kafka.sasl.mechanism" : "PLAIN",
    "kafka.ssl.endpoint.identification.algorithm" :"https",
    "kafka.sasl.jaas.config" : 
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='MAB2I5Y4QV7PMJPA' password='cflt/QktWCHgyWPsmVvYOTjFSOpkQszdxO7h9sgGf56MmH46hozKty2abFQxTAZw';"



}

In [0]:
df = (
    spark.readStream
        .format("kafka")
        .options(**myconfigurations)
        .load()
)


In [0]:
df = df.selectExpr(
    "CAST(key AS string) AS key",
    "CAST(value AS string) AS value",
    "topic",
    "partition",
    "offset",
    "timestamp",
    "timestampType"
)

In [0]:
#display(converted_orders_df)

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# schema = StructType([
#     StructField("key", StringType()),
#     StructField("value", StringType()),
#     StructField("event_ts", TimestampType())
# ])

# json_df = df.withColumn("data" , from_json(col("value").cast("string"), schema))
# json_df=  json_df.select(col("data.*"))


In [0]:
def upsert_to_delta(microBatchDF, batchId):
    print(f"Processing batch: {batchId}")
    
    (microBatchDF
        .write
        .format("delta")
        .mode("overWrite")
        .saveAsTable("namaste_catalog.vineetdb.kafka_streaming")
    )

In [0]:
try:
    (df.writeStream
     .outputMode("append")
     .trigger(once=True)
    .foreachBatch(upsert_to_delta)
    .option('checkpointLocation','/Volumes/namaste_catalog/vineetdb/testvolume/checkpoint/')
    .start()
    )
except Exception as e:
    print(f'please connect to the right zooker', {e})